# Adaptive Inventory & Pricing with Tabular Reinforcement Learning

**Group Project 13**  
**Members:** Jiayi Zhuo, Keyang Li, Rongze Gao, Zhexi Wang  
**Course:** BU.520.750.51.SP26  

This notebook is the complete technical report for the project. It documents the motivation,
data-processing pipeline, finite MDP design, tabular RL methods, hyperparameter tuning,
baseline comparison, and final empirical analysis. All reported results are generated by
the repository scripts and saved under `reports/`.

## 1. Research Context and Literature

The project studies joint retail pricing and replenishment as a sequential decision problem.
This framing is motivated by classical reinforcement learning, where an agent learns a policy
through interaction with a Markov Decision Process (Sutton & Barto, 2018). Q-learning is a
model-free temporal-difference method for learning action values in controlled Markovian
domains (Watkins & Dayan, 1992), while SARSA provides an on-policy temporal-difference
alternative that evaluates the behavior policy under exploration (Rummery & Niranjan, 1994;
Sutton & Barto, 2018).

The retail data foundation is the M5 Forecasting Accuracy dataset. The M5 competition used
Walmart unit-sales data with daily product-store observations, sell prices, and calendar
information, making it a credible source for demand calibration (Makridakis et al., 2022;
Kaggle, 2024; Zenodo, 2024). Because M5 does not include true on-hand inventory or lost
demand, this project uses M5 to calibrate demand and builds an explicit simulator for
replenishment, holding cost, stockout cost, and price response. This choice is consistent with
the operations literature, where inventory control is commonly represented as an MDP with
ordering, holding, and shortage costs (Gijsbrechts et al., 2022; Mahajan, n.d.; NEASQC, 2024).
Dynamic pricing studies similarly motivate reinforcement learning because prices influence
future demand and revenue trajectories rather than only one-period outcomes (Apte et al.,
2024).

## 2. Data Source and Selected SKU

The experiment selected one M5 product-store time series:

- Item: `FOODS_3_252`
- Store: `TX_1`
- State: `TX`
- Department/category: `FOODS_3` / `FOODS`
- Mean daily sales: `29.530`
- Nonzero-sales ratio: `0.996`
- Median sell price: `$1.48`
- Price coefficient of variation: `0.032`

The preprocessing script chooses a series with enough nonzero demand and nontrivial price
information so that the simulator does not collapse into a degenerate "never order" problem.
The chronological split preserves time order: train data calibrates demand, validation data
tunes hyperparameters, and test episodes evaluate final policies.

In [1]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

reports = Path('../reports') if Path('../reports').exists() else Path('reports')
tables = reports / 'tables'
figures = reports / 'figures'

sales_by_split = pd.read_csv(tables / 'sales_by_split.csv')
preview = pd.read_csv(tables / 'selected_series_preview.csv')
evaluation_summary = pd.read_csv(tables / 'evaluation_summary.csv')
tuning_results = pd.read_csv(tables / 'hyperparameter_tuning_results.csv')
training_history = pd.read_csv(tables / 'training_history.csv')
evaluation_episodes = pd.read_csv(tables / 'evaluation_episodes.csv')
sales_by_split

,split,count,mean,std,min,median,max
0,test,292,32.205479,13.340849,1,31.0,109
1,train,1358,29.734904,15.035957,0,28.0,105
2,validation,291,25.890034,12.384707,0,25.0,85


### Split-level Sales Summary

| split      |   count |    mean |     std |   min |   median |   max |
|:-----------|--------:|--------:|--------:|------:|---------:|------:|
| test       |     292 | 32.2055 | 13.3408 |     1 |       31 |   109 |
| train      |    1358 | 29.7349 | 15.036  |     0 |       28 |   105 |
| validation |     291 | 25.89   | 12.3847 |     0 |       25 |    85 |

### Processed Series Preview

| date       |   sales |   sell_price | calendar_type   | demand_signal   | split   |
|:-----------|--------:|-------------:|:----------------|:----------------|:--------|
| 2011-01-29 |      45 |         1.48 | weekend         | high            | train   |
| 2011-01-30 |      33 |         1.48 | weekend         | high            | train   |
| 2011-01-31 |      23 |         1.48 | weekday         | high            | train   |
| 2011-02-01 |      13 |         1.48 | event_or_snap   | high            | train   |
| 2011-02-02 |       8 |         1.48 | weekday         | normal          | train   |
| 2011-02-03 |      15 |         1.48 | event_or_snap   | normal          | train   |
| 2011-02-04 |      21 |         1.48 | weekday         | low             | train   |
| 2011-02-05 |      26 |         1.48 | event_or_snap   | low             | train   |
| 2011-02-06 |      21 |         1.48 | event_or_snap   | low             | train   |
| 2011-02-07 |      23 |         1.48 | event_or_snap   | low             | train   |
| 2011-02-08 |      22 |         1.48 | weekday         | low             | train   |
| 2011-02-09 |      19 |         1.48 | event_or_snap   | low             | train   |

### Data Overview Figure

![Data overview](../reports/figures/data_overview.png)

## 3. MDP Design

The environment is a finite MDP with `405` states and `12`
actions, for `4860` tabular state-action values. This is intentionally
small enough for tabular RL and large enough to represent the core operational tradeoffs.

The state is:

`s_t = (inventory_bin, demand_signal_bin, price_tier, calendar_type, pipeline_bin)`

Inventory is discretized into stockout, low, medium, high, and excess. Demand signal is based
on lagged seven-day rolling demand and discretized into low, normal, and high. Price tier is
discount, regular, or premium. Calendar type distinguishes weekday, weekend, and event/SNAP
contexts. Pipeline inventory records whether replenishment is already arriving soon.

The action is:

`a_t = (price_choice, order_quantity)`

Price choices are discount, regular, and premium. Order choices are none, small, medium, and
large. The selected simulator parameters are:

- Capacity: `140` units
- Lead time: `2` days
- Order quantities: `[0, 21, 42, 70]`
- Discount price multiplier: `0.9`
- Premium price multiplier: `1.1`
- Discount demand lift: `0.15`
- Premium demand drop: `0.15`

The reward is daily operating profit: sales revenue minus procurement cost, fixed ordering
cost, holding cost, stockout penalty, excess-inventory penalty, and price-change penalty.
This reward avoids the common mistake of optimizing revenue alone, which can over-discount
or over-order, and also avoids a service-insensitive profit objective by penalizing lost sales.

## 4. Training and Hyperparameter Optimization

The project compares tabular Q-learning and tabular SARSA. A grid search was run over learning
rate, discount factor, epsilon decay, and minimum epsilon. Each configuration was trained across
multiple seeds and evaluated on validation episodes. The best configuration for each algorithm
was then retrained for the final training budget and evaluated on held-out test episodes.

This validation-first design prevents selecting hyperparameters using the final test results.

### Top Hyperparameter Configurations

| algorithm   |   alpha |   gamma |   epsilon_decay |   epsilon_min |   episodes |   seeds |   validation_return_mean |   validation_return_std |   validation_fill_rate_mean |
|:------------|--------:|--------:|----------------:|--------------:|-----------:|--------:|-------------------------:|------------------------:|----------------------------:|
| q_learning  |     0.2 |    0.99 |           0.999 |          0.03 |        600 |       3 |                  985.12  |                 17.6972 |                    0.941882 |
| q_learning  |     0.2 |    0.99 |           0.999 |          0.05 |        600 |       3 |                  985.12  |                 17.6972 |                    0.941882 |
| q_learning  |     0.2 |    0.95 |           0.999 |          0.03 |        600 |       3 |                  739.554 |                115.693  |                    0.854647 |
| q_learning  |     0.2 |    0.95 |           0.999 |          0.05 |        600 |       3 |                  739.554 |                115.693  |                    0.854647 |
| q_learning  |     0.2 |    0.99 |           0.996 |          0.05 |        600 |       3 |                  615.773 |                109.175  |                    0.828088 |
| q_learning  |     0.2 |    0.99 |           0.996 |          0.03 |        600 |       3 |                  615.773 |                109.175  |                    0.828088 |
| q_learning  |     0.2 |    0.95 |           0.992 |          0.05 |        600 |       3 |                  520.202 |                157.025  |                    0.797897 |
| q_learning  |     0.2 |    0.99 |           0.992 |          0.05 |        600 |       3 |                  505.121 |                 84.737  |                    0.795794 |
| q_learning  |     0.2 |    0.95 |           0.996 |          0.05 |        600 |       3 |                  464.631 |                180.505  |                    0.788722 |
| q_learning  |     0.2 |    0.95 |           0.996 |          0.03 |        600 |       3 |                  464.631 |                180.505  |                    0.788722 |

### Training Curve

![Training curve](../reports/figures/training_curve.png)

In [2]:
training_history.groupby('algorithm')['return'].agg(['count', 'mean', 'std', 'min', 'max']).round(3)

,count,mean,std,min,max
algorithm,,,,,
q_learning,3000,735.917,301.723,-1482.238,1488.096
sarsa,3000,334.353,293.045,-1796.370,1232.280


## 5. Baselines

The learned policies are compared against four interpretable baselines:

1. **Random valid action**, a lower-bound sanity check.
2. **Static regular price + reorder point**, a standard inventory-control heuristic.
3. **Regular + average demand order-up-to**, a demand-planning baseline.
4. **Inventory markdown heuristic**, which discounts high inventory and raises price under scarcity.

These baselines make the comparison stronger than only showing that RL beats random behavior.

## 6. Test Evaluation Results

| policy                               |   return_mean |   return_std |   fill_rate_mean |   fill_rate_std |   stockout_units_mean |   stockout_units_std |   avg_inventory_mean |   avg_inventory_std |   avg_order_quantity_mean |   avg_order_quantity_std |   price_changes_mean |   price_changes_std |
|:-------------------------------------|--------------:|-------------:|-----------------:|----------------:|----------------------:|---------------------:|---------------------:|--------------------:|--------------------------:|-------------------------:|---------------------:|--------------------:|
| Q Learning                           |      791.717  |      187.608 |         0.881702 |      0.066142   |               398.8   |             276.919  |              44.6013 |            13.2627  |                   23.7776 |                 1.93699  |              74.9    |             5.98654 |
| Sarsa                                |      461.01   |      139.955 |         0.76897  |      0.0441972  |               580.542 |             190.643  |              16.0122 |             2.4881  |                   15.5001 |                 1.40541  |              70.8917 |             7.38702 |
| Static regular + reorder point       |      337.641  |      462.28  |         0.753701 |      0.104554   |               866.2   |             553.941  |              22.2763 |             5.55187 |                   19.4688 |                 1.74752  |               0      |             0       |
| Inventory markdown heuristic         |      282.231  |      141.191 |         0.69729  |      0.041203   |               652.467 |             155.303  |              11.7429 |             1.11852 |                   12.0517 |                 0.454335 |              30.3667 |             4.79484 |
| Regular + average demand order-up-to |      152.273  |      752.889 |         0.99753  |      0.00460839 |                 8.825 |              17.1468 |             102.996  |             9.97247 |                   38.4742 |                 3.2696   |               0      |             0       |
| Random valid action                  |       36.8964 |      529.11  |         0.923945 |      0.0621313  |               278.983 |             258.595  |              78.0249 |            19.092   |                   33.7886 |                 2.67168  |              79.6833 |             4.96709 |

### Test Return Comparison

![Evaluation returns](../reports/figures/evaluation_returns.png)

### Operational Metrics

![Operational metrics](../reports/figures/evaluation_operational_metrics.png)

The best average test return was achieved by **Q Learning** with a mean cumulative
reward of `791.72` and a mean fill rate of
`0.882`. The fill-rate metric is important because a policy can
increase short-term profit by avoiding inventory, but that behavior would create stockouts and
poor service. The combined profit and service analysis therefore gives a more complete view of
policy quality.

In [3]:
evaluation_episodes.groupby('policy')[['return', 'fill_rate', 'stockout_units', 'avg_inventory']].describe().round(3)

return                              \
                                      count     mean      std       min   
policy                                                                    
Inventory markdown heuristic          120.0  282.231  141.191  -549.777   
Q Learning                            120.0  791.717  187.608   230.882   
Random valid action                   120.0   36.896  529.110 -1233.162   
Regular + average demand order-up-to  120.0  152.273  752.889  -946.302   
Sarsa                                 120.0  461.010  139.955  -128.195   
Static regular + reorder point        120.0  337.641  462.280 -1094.682   

                                                                           \
                                          25%      50%      75%       max   
policy                                                                      
Inventory markdown heuristic          211.315  300.100  367.121   564.742   
Q Learning                            683.933  797.857  929.593  1187.891   
Random valid action                  -363.810   11.629  398.511  1252.476   
Regular + average demand order-up-to -452.104   38.645  614.143  2116.538   
Sarsa                                 396.014  485.914  549.077   763.815   
Static regular + reorder point         51.493  444.438  737.063   946.886   

                                     fill_rate         ... stockout_units  \
                                         count   mean  ...            75%   
policy                                                 ...                  
Inventory markdown heuristic             120.0  0.697  ...         718.25   
Q Learning                               120.0  0.882  ...         591.75   
Random valid action                      120.0  0.924  ...         455.00   
Regular + average demand order-up-to     120.0  0.998  ...          11.25   
Sarsa                                    120.0  0.769  ...         701.00   
Static regular + reorder point           120.0  0.754  ...        1191.75   

                                             avg_inventory                   \
                                         max         count     mean     std   
policy                                                                        
Inventory markdown heuristic          1592.0         120.0   11.743   1.119   
Q Learning                            1411.0         120.0   44.601  13.263   
Random valid action                   1000.0         120.0   78.025  19.092   
Regular + average demand order-up-to   110.0         120.0  102.996   9.972   
Sarsa                                 1588.0         120.0   16.012   2.488   
Static regular + reorder point        2494.0         120.0   22.276   5.552   

                                                                        \
                                         min     25%      50%      75%   
policy                                                                   
Inventory markdown heuristic           9.275  10.983   11.775   12.567   
Q Learning                            12.633  35.698   45.296   53.808   
Random valid action                   23.633  65.271   77.329   94.865   
Regular + average demand order-up-to  73.708  97.133  104.962  111.165   
Sarsa                                  8.750  14.138   16.062   17.794   
Static regular + reorder point         9.608  18.692   22.400   26.494   

                                               
                                          max  
policy                                         
Inventory markdown heuristic           15.192  
Q Learning                             76.242  
Random valid action                   112.242  
Regular + average demand order-up-to  115.800  
Sarsa                                  22.025  
Static regular + reorder point         34.150  

[6 rows x 32 columns]

## 7. Learned Policy Interpretation

![Q-learning policy heatmap](../reports/figures/q_learning_policy_heatmap.png)

![SARSA policy heatmap](../reports/figures/sarsa_policy_heatmap.png)

The policy heatmaps show one interpretable slice of the learned policy: weekday, regular current
price, and no pipeline inventory. The table annotations reveal how the action changes as
inventory and demand signal move from low to high. This is the main advantage of keeping the
project tabular: the final policy can be inspected directly rather than treated as a black box.

## 8. Limitations and Sensitivity

The experiment makes explicit assumptions because public M5 data does not report inventory,
replenishment orders, or lost demand. Observed sales may be censored by historical stockouts,
and historical prices are observational rather than randomized. For that reason, the project
should be interpreted as a tabular RL simulator calibrated by real retail data, not as a causal
estimate of Walmart's true pricing response.

We therefore evaluate sensitivity to price elasticity, lead time, holding cost, and stockout
penalty. In this analysis, the learned Q-tables are kept fixed and evaluated under perturbed
simulator assumptions. This does not replace retraining under each scenario, but it reveals how
robust the learned policies are when the operating assumptions change.

### Sensitivity Results

| scenario        | scenario_type    | policy                               |     return |   fill_rate |   stockout_units |   avg_inventory |
|:----------------|:-----------------|:-------------------------------------|-----------:|------------:|-----------------:|----------------:|
| base            | base             | Q Learning                           |  777.011   |    0.888223 |         368.25   |        46.7729  |
| base            | base             | Sarsa                                |  472.738   |    0.774283 |         569.417  |        16.3576  |
| base            | base             | Random valid action                  |   30.6171  |    0.916512 |         311.167  |        75.7217  |
| base            | base             | Static regular + reorder point       |  317.334   |    0.741036 |         910.517  |        21.4275  |
| base            | base             | Regular + average demand order-up-to |  241.039   |    0.997155 |          10.2333 |       102.081   |
| base            | base             | Inventory markdown heuristic         |  315.243   |    0.706448 |         622.65   |        11.7531  |
| elasticity_low  | price_elasticity | Q Learning                           |  712.028   |    0.861562 |         489.15   |        45.1889  |
| elasticity_low  | price_elasticity | Sarsa                                |  400.085   |    0.738687 |         768.483  |        14.4235  |
| elasticity_low  | price_elasticity | Random valid action                  |   29.9209  |    0.917298 |         300.367  |        76.2847  |
| elasticity_low  | price_elasticity | Static regular + reorder point       |  317.334   |    0.741036 |         910.517  |        21.4275  |
| elasticity_low  | price_elasticity | Regular + average demand order-up-to |  241.039   |    0.997155 |          10.2333 |       102.081   |
| elasticity_low  | price_elasticity | Inventory markdown heuristic         |  -26.4421  |    0.617692 |        1032.93   |         9.96875 |
| elasticity_high | price_elasticity | Q Learning                           |  781.902   |    0.913673 |         265.283  |        50.5081  |
| elasticity_high | price_elasticity | Sarsa                                |  597.233   |    0.838677 |         346.833  |        19.1514  |
| elasticity_high | price_elasticity | Random valid action                  |    8.76055 |    0.879489 |         478.333  |        67.6475  |
| elasticity_high | price_elasticity | Static regular + reorder point       |  317.334   |    0.741036 |         910.517  |        21.4275  |
| elasticity_high | price_elasticity | Regular + average demand order-up-to |  241.039   |    0.997155 |          10.2333 |       102.081   |
| elasticity_high | price_elasticity | Inventory markdown heuristic         |  478.453   |    0.783462 |         388.567  |        13.9824  |
| lead_time_1     | lead_time        | Q Learning                           | 1113.66    |    0.967699 |         107.667  |        63.445   |
| lead_time_1     | lead_time        | Sarsa                                |  876.049   |    0.879061 |         313.317  |        22.9331  |

![Sensitivity returns](../reports/figures/sensitivity_returns.png)

## 9. Reproducibility

Main commands:

```powershell
python scripts/download_m5.py
python scripts/run_experiments.py --tuning-episodes 600 --final-episodes 3000 --evaluation-episodes 120
python scripts/build_report_notebook.py
pytest
```

CI uses a synthetic smoke dataset to keep GitHub Actions fast and independent of the external
M5 download. The full local report uses the real M5-derived processed subset.

## References

Apte, M., Kale, K., Datar, P., & Deshmukh, P. (2024). *Dynamic retail pricing via Q-learning:
A reinforcement learning framework for enhanced revenue management*. arXiv.
https://arxiv.org/abs/2411.18261

Gijsbrechts, J., Boute, R. N., Van Mieghem, J. A., & Zhang, D. (2022). Deep reinforcement
learning for inventory control: A roadmap. *European Journal of Operational Research, 298*(2),
401-412. https://doi.org/10.1016/j.ejor.2021.07.016

Kaggle. (2024). *M5 Forecasting - Accuracy*. https://www.kaggle.com/competitions/m5-forecasting-accuracy/data

Mahajan, A. (n.d.). *Inventory management revisited*. https://adityam.github.io/stochastic-control/mdps/inventory-management-revisited.html

Makridakis, S., Spiliotis, E., & Assimakopoulos, V. (2022). The M5 competition: Background,
organization, and implementation. *International Journal of Forecasting, 38*(4), 1325-1336.
https://doi.org/10.1016/j.ijforecast.2021.07.007

NEASQC. (2024). *Reinforcement learning for inventory management*.
https://www.neasqc.eu/use-case/reinforcement-learning-for-inventory-management/

Rummery, G. A., & Niranjan, M. (1994). *On-line Q-learning using connectionist systems*
(Technical Report CUED/F-INFENG/TR 166). Cambridge University Engineering Department.

Sutton, R. S., & Barto, A. G. (2018). *Reinforcement learning: An introduction* (2nd ed.).
MIT Press.

Watkins, C. J. C. H., & Dayan, P. (1992). Q-learning. *Machine Learning, 8*, 279-292.
https://doi.org/10.1007/BF00992698

Zenodo. (2024). *M5 Forecasting Accuracy dataset* (Version v1) [Data set].
https://doi.org/10.5281/zenodo.12636070